# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue ranks content records for human review using observed performance signals from the March 2026 dataset.

The queue uses simple reason codes so that each recommendation has a clear explanation. A high-priority record is a review candidate, not an automatic instruction to change content.

Priority is based on observed impressions, clicks, average position, and the model's predicted probability of receiving a click. The ranking is intended to help reviewers decide where to investigate first.

Suggested reason codes:

* `HIGH_IMPRESSIONS_LOW_CLICKS` — substantial observed impressions with zero or very low clicks.
* `WEAK_POSITION` — observed average position is relatively weak and may warrant review.
* `HIGH_PRIORITY_REVIEW` — multiple observed signals indicate that the record deserves closer inspection.
* `MONITOR` — no strong action signal; keep under observation.

These reason codes are prioritization aids, not causal diagnoses.


In [2]:
import pandas as pd

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Rows:", len(df_march))

Rows: 9841378


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Use the March 2026 dataset
playbook_df = df_march.copy()

# Basic observed signals
playbook_df["gsc_clicks"] = playbook_df["gsc_clicks"].fillna(0)
playbook_df["gsc_impressions"] = playbook_df["gsc_impressions"].fillna(0)
playbook_df["gsc_avg_position"] = playbook_df["gsc_avg_position"].fillna(0)

# Avoid division by zero
playbook_df["ctr_observed"] = np.where(
    playbook_df["gsc_impressions"] > 0,
    playbook_df["gsc_clicks"] / playbook_df["gsc_impressions"],
    0
)

# Simple review signals
playbook_df["high_impressions"] = (
    playbook_df["gsc_impressions"] >=
    playbook_df["gsc_impressions"].quantile(0.75)
)

playbook_df["low_clicks"] = (
    playbook_df["gsc_clicks"] <= 1
)

playbook_df["weak_position"] = (
    playbook_df["gsc_avg_position"] > 10
)

# Reason code
playbook_df["reason_code"] = "MONITOR"

playbook_df.loc[
    playbook_df["high_impressions"] & playbook_df["low_clicks"],
    "reason_code"
] = "HIGH_IMPRESSIONS_LOW_CLICKS"

playbook_df.loc[
    (playbook_df["reason_code"] == "MONITOR") &
    playbook_df["weak_position"],
    "reason_code"
] = "WEAK_POSITION"

# Priority score for review ordering
playbook_df["priority_score"] = (
    playbook_df["high_impressions"].astype(int) * 2
    + playbook_df["low_clicks"].astype(int)
    + playbook_df["weak_position"].astype(int)
)

# Highest-priority records first
ranked_queue = playbook_df.sort_values(
    ["priority_score", "gsc_impressions"],
    ascending=[False, False]
).head(100).copy()

print("Ranked queue created.")
print("Rows in queue:", len(ranked_queue))
print("\nReason-code counts:")
print(ranked_queue["reason_code"].value_counts())

display(
    ranked_queue[
        [
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr_observed",
            "reason_code",
            "priority_score"
        ]
    ].head(20)
)

Ranked queue created.
Rows in queue: 100

Reason-code counts:
reason_code
HIGH_IMPRESSIONS_LOW_CLICKS    100
Name: count, dtype: int64


,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr_observed,reason_code,priority_score
2775255,2026-03-09,9409,1,32.640344,0.000106,HIGH_IMPRESSIONS_LOW_CLICKS,4
1559139,2026-03-06,7459,1,33.247754,0.000134,HIGH_IMPRESSIONS_LOW_CLICKS,4
9118396,2026-03-30,7026,0,28.103473,0.000000,HIGH_IMPRESSIONS_LOW_CLICKS,4
6150102,2026-03-19,6892,0,10.337057,0.000000,HIGH_IMPRESSIONS_LOW_CLICKS,4
2776382,2026-03-09,6541,0,36.109005,0.000000,HIGH_IMPRESSIONS_LOW_CLICKS,4
2948808,2026-03-10,6396,0,34.615228,0.000000,HIGH_IMPRESSIONS_LOW_CLICKS,4
2036550,2026-03-07,6106,1,10.047494,0.000164,HIGH_IMPRESSIONS_LOW_CLICKS,4
3808812,2026-03-11,6098,0,34.625779,0.000000,HIGH_IMPRESSIONS_LOW_CLICKS,4
1920763,2026-03-08,6036,1,39.571571,0.000166,HIGH_IMPRESSIONS_LOW_CLICKS,4
3810291,2026-03-11,5996,1,35.924783,0.000167,HIGH_IMPRESSIONS_LOW_CLICKS,4


## 2. Intended use and limits

The playbook is intended for SEO/content reviewers who need a ranked starting point for investigation.

It can help reviewers:

* prioritize high-impression, low-click records;
* identify records with relatively weak observed position;
* focus limited review time on a smaller queue;
* use reason codes to understand why a record was ranked.

The playbook does not determine whether content should be rewritten, deleted, redirected, or published automatically.

Its limits are important. The underlying data is observational and limited to the March 2026 reporting window. The ranking does not establish that changing a page will cause better performance. It also does not include all factors that may affect clicks, such as search intent, SERP features, brand effects, content quality, or business priorities.

The queue should therefore be treated as a prioritization tool for human review, not as an autonomous content optimizer.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("INTENDED USE")
print("-" * 50)
print("Primary user: SEO/content reviewer")
print("Purpose: prioritize records for investigation")
print("Automation: ranking and reason codes only")
print("Content changes: human decision required")

print("\nKEY LIMITS")
print("-" * 50)
print("Data window: March 2026")
print("Data type: observational")
print("Causal conclusion: not supported")
print("Automatic rewrite/delete/redirect: not allowed")
print("SERP context and search intent: require human review")

INTENDED USE
--------------------------------------------------
Primary user: SEO/content reviewer
Purpose: prioritize records for investigation
Automation: ranking and reason codes only
Content changes: human decision required

KEY LIMITS
--------------------------------------------------
Data window: March 2026
Data type: observational
Causal conclusion: not supported
Automatic rewrite/delete/redirect: not allowed
SERP context and search intent: require human review


## 3. Human review + the no-go list

Every ranked record must be reviewed by a person before any content action is taken.

### Human-review checklist

Before acting on a recommendation, the reviewer should check:

1. **Search intent** — Does the page match what users are searching for?
2. **SERP context** — What competing results, features, and search conditions are present?
3. **Content quality** — Is the content accurate, useful, current, and sufficiently complete?
4. **Business importance** — Does the page support an important product, topic, or business goal?
5. **Freshness** — Is the content outdated, and is there evidence that a refresh is appropriate?
6. **Observed data quality** — Are the relevant GSC/GA4 signals actually available rather than simply missing?
7. **Change risk** — Could changing the page harm useful existing traffic or rankings?

### No-go list

The system should **not** automatically:

* rewrite or publish content;
* delete pages;
* redirect URLs;
* change titles or metadata;
* change keywords or search intent;
* make claims about causation;
* decide that a page is low quality;
* apply a recommendation without human review.

The model ranks review candidates. A human makes the final content decision.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("HUMAN REVIEW REQUIRED")
print("-" * 50)

review_checks = [
    "Search intent",
    "SERP context",
    "Content quality",
    "Business importance",
    "Freshness",
    "GSC/GA4 data availability",
    "Change risk"
]

for i, check in enumerate(review_checks, 1):
    print(f"{i}. {check}")

print("\nNO-GO AUTOMATION LIST")
print("-" * 50)

no_go_actions = [
    "Automatic content rewriting or publishing",
    "Automatic page deletion",
    "Automatic URL redirects",
    "Automatic title/metadata changes",
    "Automatic keyword or intent changes",
    "Causal claims from model output",
    "Automatic low-quality classification",
    "Taking action without human review"
]

for action in no_go_actions:
    print("NO:", action)

print("\nFinal rule: model output = review priority, not automatic action.")

HUMAN REVIEW REQUIRED
--------------------------------------------------
1. Search intent
2. SERP context
3. Content quality
4. Business importance
5. Freshness
6. GSC/GA4 data availability
7. Change risk

NO-GO AUTOMATION LIST
--------------------------------------------------
NO: Automatic content rewriting or publishing
NO: Automatic page deletion
NO: Automatic URL redirects
NO: Automatic title/metadata changes
NO: Automatic keyword or intent changes
NO: Causal claims from model output
NO: Automatic low-quality classification
NO: Taking action without human review

Final rule: model output = review priority, not automatic action.


## 4. Monitoring / retrain triggers

The playbook should be monitored because content performance patterns can change over time.

Monitoring should focus on:

Data availability: check whether GSC and GA4 coverage changes materially.
Queue composition: monitor whether the same reason codes dominate the queue over time.
Model performance: when a later period has observed click outcomes, compare performance with the current validation results.
Feature drift: check whether the distributions of impressions, position, sessions, or AI sessions change substantially.
Recommendation usefulness: collect reviewer feedback on whether ranked records were useful review candidates.

A retrain or review should be considered when there is sustained feature drift, meaningful changes in data coverage, a material decline in measured model performance on a later labeled period, or a substantial change in the content/SEO environment.

A single unusual day should not automatically trigger retraining. Retraining decisions should be based on sustained evidence and human review.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("MONITORING CHECKLIST")
print("-" * 50)

monitoring_items = [
    "GSC/GA4 data availability",
    "Reason-code distribution",
    "Model performance on later labeled data",
    "Feature distribution drift",
    "Reviewer usefulness feedback"
]

for item in monitoring_items:
    print("-", item)

print("\nRETRAIN / REVIEW TRIGGERS")
print("-" * 50)

triggers = [
    "Sustained feature drift",
    "Meaningful change in data coverage",
    "Material performance decline on later labeled data",
    "Substantial change in the SEO/content environment",
    "Repeated reviewer feedback that the queue is no longer useful"
]

for trigger in triggers:
    print("TRIGGER:", trigger)

print("\nRule: one unusual day alone does not trigger retraining.")

MONITORING CHECKLIST
--------------------------------------------------
- GSC/GA4 data availability
- Reason-code distribution
- Model performance on later labeled data
- Feature distribution drift
- Reviewer usefulness feedback

RETRAIN / REVIEW TRIGGERS
--------------------------------------------------
TRIGGER: Sustained feature drift
TRIGGER: Meaningful change in data coverage
TRIGGER: Material performance decline on later labeled data
TRIGGER: Substantial change in the SEO/content environment
TRIGGER: Repeated reviewer feedback that the queue is no longer useful

Rule: one unusual day alone does not trigger retraining.


## 5. Exports for the paper

The ranked action queue is exported so it can be reviewed and referenced in the recommendations section of the paper.

The export contains only the fields needed for prioritization and review. Client names, URLs, and private search queries are not included.

The exported queue should be treated as a human-review worklist rather than an automated content-action list.

### Archetype → action mapping

| Observed archetype                | Suggested human-review action                                                       |
| --------------------------------- | ----------------------------------------------------------------------------------- |
| High impressions + low clicks     | Review search intent, SERP context, title/snippet, and content alignment            |
| Weak observed position            | Review content depth, competition, internal linking, and search intent              |
| Monitor                           | Continue monitoring before making a content change                                  |
| Older content with recent refresh | Review whether the refresh pattern is consistent with improved observed performance |

These are review actions, not automatic decisions.

### Decay and refresh insight

The research observed that content health generally declined at older ages, while older content that had been refreshed could show stronger observed performance. In particular, the 365+ day refreshed group showed higher observed health than the comparable stale group. This finding is observational and should not be interpreted as proof that refreshing a page will cause improvement.

### Cost/value thinking

Review capacity is limited, so the queue should help reviewers spend time where the potential value of investigation is higher. High-impression, low-click records are therefore useful early review candidates because they represent substantial observed visibility with limited observed clicks.

Traffic-value calculations such as clicks multiplied by CPC may be used as a proxy for captured traffic value, but they should not be treated as revenue or ROI without additional business data.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

export_queue = ranked_queue[
    [
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr_observed",
        "reason_code",
        "priority_score"
    ]
].copy()

queue_path = "work/outputs/ml10_ranked_action_queue.csv"
export_queue.to_csv(queue_path, index=False)

reason_summary = (
    export_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

summary_path = "work/outputs/ml10_reason_code_summary.csv"
reason_summary.to_csv(summary_path, index=False)

print("EXPORTS CREATED")
print("-" * 50)
print(queue_path)
print(summary_path)

print("\nExported queue rows:", len(export_queue))
print("Exported columns:", list(export_queue.columns))

EXPORTS CREATED
--------------------------------------------------
work/outputs/ml10_ranked_action_queue.csv
work/outputs/ml10_reason_code_summary.csv

Exported queue rows: 100
Exported columns: ['content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr_observed', 'reason_code', 'priority_score']


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.